# Day 13 - Lakehouse Table Delta vs Parquet

**Topic:** Lakehouse Table Delta vs Parquet  
**Main environment:** Microsoft Fabric Notebook + Fabric Lakehouse  
**Focus:** เข้าใจความต่างระหว่าง file-based processing แบบ Parquet path กับ Lakehouse table แบบ Delta table ที่อ่านเขียนผ่าน table name และมี table metadata / transaction log ช่วยจัดการความน่าเชื่อถือของข้อมูล

วันนี้เราจะเริ่มเข้า Phase 3 ของ challenge คือ Lakehouse table concepts โดยยังไม่ลงลึกเรื่อง MERGE, history, time travel หรือ table maintenance มากเกินไป เป้าหมายหลักคือเห็นภาพว่าในงานจริงเราไม่ได้ทำงานกับไฟล์ที่วางอยู่ใน folder/path เฉย ๆ แต่ต้องเข้าใจว่าเมื่อไรควรอ่านเขียนเป็น file path และเมื่อไรควรใช้ Lakehouse table / Delta table


## Goal of the day

1. แยกให้ออกว่า **Parquet files** กับ **Lakehouse table / Delta table** ต่างกันอย่างไร
2. อธิบายได้ว่า Delta table ไม่ใช่แค่ file format แต่เป็น table layer ที่มี data files, transaction log และ metadata
3. เขียนข้อมูลเป็น Parquet path และอ่านกลับด้วย `spark.read.parquet()` ได้
4. เขียนข้อมูลเป็น Delta Lakehouse table ด้วย `saveAsTable()` และอ่านกลับด้วย `spark.read.table()` ได้
5. เริ่มเข้าใจว่า production pipeline ควรใช้ reliable table มากกว่า file path เมื่อข้อมูลนั้นเป็น shared dataset ที่หลายงานต้องอ่านต่อ


## Why it matters in production

ใน production pipeline ข้อมูลมักไม่ได้ถูกใช้ครั้งเดียวแล้วจบ แต่ถูกอ่านต่อโดย notebook, Spark job, SQL endpoint, BI report หรือ pipeline step อื่น ๆ ดังนั้นรูปแบบการเก็บข้อมูลจึงมีผลต่อ reliability และ maintainability มาก

- Parquet เป็น columnar file format ที่เหมาะกับ analytics และการเก็บข้อมูลเป็น files
- แต่ Parquet file path อย่างเดียวไม่มี table transaction log หรือ table metadata ที่ช่วย track operation ระดับ table
- Delta table ใช้ Parquet data files เป็นฐาน แล้วเพิ่ม `_delta_log` เพื่อ track transaction, schema และ table version concept
- Lakehouse table ช่วยให้เราอ่านข้อมูลด้วย table name เช่น `spark.read.table("table_name")` แทนการจำ path เองทุกครั้ง
- ใน Microsoft Fabric Lakehouse การทำงานกับ table โดยทั่วไปจะผูกกับ Delta table format เพื่อให้ใช้งานกับ Fabric experiences อื่นได้ดีขึ้น

Production takeaway:

> ไฟล์ข้อมูลที่อ่านเขียนผ่าน file path เหมาะกับ raw file landing หรือ temporary output แต่ curated dataset ที่ต้องถูกใช้ต่อควรเริ่มคิดเป็น Lakehouse table / Delta table ตั้งแต่แรก


## Key concepts

| Concept | Meaning |
|---|---|
| Parquet | Columnar file format ที่นิยมใช้กับ analytics workload |
| File path access | การอ่านเขียนข้อมูลโดยชี้ไปที่ path เช่น `Files/.../transactions_parquet` |
| Lakehouse table | Table ที่ถูก register ใน Lakehouse catalog/metastore และอ่านด้วย table name ได้ |
| Delta table | Table format ที่เก็บ data files เป็น Parquet และมี `_delta_log` สำหรับ transaction log |
| Transaction log | Log ที่บันทึก metadata ของ operation ระดับ table เช่น add/remove files และ schema |
| Table metadata | ข้อมูลประกอบ table เช่น table name, schema, location และ format |
| Managed table | Table ที่ Lakehouse/Spark จัดการ metadata และ location ให้เป็นหลัก |
| External table | Table metadata ที่ชี้ไปยัง data location ภายนอกหรือ path ที่เราควบคุมเองมากขึ้น |
| ACID concept | แนวคิดเรื่อง atomic, consistent, isolated, durable operation ในระดับ table |


## Concept explanation

### Parquet คืออะไร?

Parquet คือ file format แบบ columnar เหมาะกับงาน analytics เพราะสามารถอ่านเฉพาะ columns ที่ต้องใช้ได้ดี เช่น query ต้องการแค่ `status` และ `amount` ก็ไม่จำเป็นต้องอ่านทุก column แบบ row-based file

ใน PySpark เราสามารถเขียน Parquet ได้แบบนี้:

```python
df.write.mode("overwrite").parquet("Files/some_path")
```

และอ่านกลับด้วย:

```python
spark.read.parquet("Files/some_path")
```

จุดสำคัญคือ pattern นี้อ้างอิงด้วย **path** เป็นหลัก ไม่ใช่ table name

### Delta table คืออะไร?

Delta table คือ table format ที่ใช้ Parquet files เป็น data files แต่เพิ่ม transaction log ไว้ใน folder `_delta_log`

พูดง่าย ๆ:

> Delta table = Parquet data files + `_delta_log` + table metadata

ดังนั้น Delta table จึงเหมาะกับ dataset ที่ต้องการความน่าเชื่อถือมากกว่าไฟล์ข้อมูลที่อ่านเขียนผ่าน file path เช่น curated table, table ที่ถูกใช้ซ้ำ, table ที่ต้องรองรับ update/merge/history ใน phase ถัดไป

### Lakehouse table ต่างจาก file path อย่างไร?

ถ้าเราอ่านข้อมูลจาก file path เราต้องรู้ว่า data อยู่ที่ไหน:

```python
spark.read.parquet("Files/day13/path")
```

แต่ถ้าเป็น Lakehouse table เราสามารถอ่านด้วย table name:

```python
spark.read.table("day13_transactions_delta")
```

หรือ query ด้วย Spark SQL:

```python
spark.sql("SELECT * FROM day13_transactions_delta")
```

สิ่งนี้ทำให้ notebook และ pipeline อ่านง่ายขึ้น และลดการ hardcode path ที่กระจายอยู่หลายจุด

### Managed table vs External table แบบสั้น ๆ

ใน lab วันนี้เราจะใช้ managed-style Lakehouse table ผ่าน `saveAsTable()` เป็นหลัก เพราะเหมาะกับ Fabric Notebook learning lab

- Managed table: ให้ Lakehouse/Spark จัดการ table metadata และ storage location หลักให้
- External table: table metadata ชี้ไปยัง path/location ที่เรากำหนดเองมากขึ้น

สำหรับ Day 13 ยังไม่ต้องลงลึก external table เพราะเป้าหมายคือแยกให้ออกก่อนว่า file path กับ Lakehouse table ใช้ต่างกันอย่างไร และควรเลือกใช้แบบไหนในสถานการณ์ใด

### ACID concept แบบ high level

ACID เป็นแนวคิดเรื่องความน่าเชื่อถือของ operation ระดับ table เช่น write สำเร็จหรือไม่สำเร็จอย่างชัดเจน และ consumer ไม่ควรเห็นข้อมูลครึ่ง ๆ กลาง ๆ จาก operation ที่ยังไม่ complete

ใน Day 13 ให้จำไว้แค่นี้ก่อน:

> Delta table เพิ่ม reliability layer เหนือ Parquet files ผ่าน transaction log

รายละเอียดเรื่อง history, time travel, recovery, optimize และ vacuum จะไปลงลึกใน Day 17-18


## Mermaid diagram: File Path vs Delta Lakehouse Table

```mermaid
flowchart LR
    A[PySpark DataFrame] --> B[Write as Parquet files]
    B --> C[Files path<br/>Files/day13/...]
    C --> D[Read by path<br/>spark.read.parquet]

    A --> E[Write as Delta table]
    E --> F[Parquet data files]
    E --> G[_delta_log]
    E --> H[Lakehouse table metadata]
    F --> I[Read by table name<br/>spark.read.table]
    G --> I
    H --> I
    I --> J[Spark SQL / Notebook / downstream consumers]
```

Key idea:

> Parquet path คือ files ที่ต้องรู้ location เอง ส่วน Delta Lakehouse table คือ table abstraction ที่มี data files, log และ metadata ให้เรียกใช้ผ่าน table name ได้


## Setup / imports


In [1]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

StatementMeta(, c54a50f4-369a-4062-9e9f-45ea83d8c97b, 3, Finished, Available, Finished, False)

## Create mock data

In [2]:
transactions_data = [
    ("T001", 101, "Bangkok", "2026-05-01", 1200.50, "success", "pos"),
    ("T002", 102, "Bangkok", "2026-05-01", 250.00, "success", "mobile_app"),
    ("T003", 103, "Chiang Mai", "2026-05-02", 3400.00, "success", "pos"),
    ("T004", 104, "Rayong", "2026-05-02", 0.00, "failed", "web"),
    ("T005", 105, "Bangkok", "2026-05-03", 780.00, "pending", "mobile_app"),
    ("T006", 106, "Phuket", "2026-05-03", 5600.00, "success", "partner_api"),
    ("T007", 107, "Bangkok", "2026-05-04", 150.00, "failed", "web"),
    ("T008", 108, "Chiang Mai", "2026-05-04", 980.00, "success", "pos"),
]

transaction_schema = T.StructType([
    T.StructField("transaction_id", T.StringType(), False),
    T.StructField("customer_id", T.IntegerType(), False),
    T.StructField("region", T.StringType(), True),
    T.StructField("transaction_date", T.StringType(), True),
    T.StructField("amount", T.DoubleType(), True),
    T.StructField("status", T.StringType(), True),
    T.StructField("source_system", T.StringType(), True),
])

df_transactions_raw = spark.createDataFrame(transactions_data, transaction_schema)

df_transactions = (
    df_transactions_raw
    .withColumn("transaction_date", F.to_date("transaction_date"))
    .withColumn("ingestion_timestamp", F.current_timestamp())
)

df_transactions.show(truncate=False)
df_transactions.printSchema()
print("Transaction count:", df_transactions.count())

StatementMeta(, c54a50f4-369a-4062-9e9f-45ea83d8c97b, 4, Finished, Available, Finished, False)

+--------------+-----------+----------+----------------+------+-------+-------------+--------------------------+
|transaction_id|customer_id|region    |transaction_date|amount|status |source_system|ingestion_timestamp       |
+--------------+-----------+----------+----------------+------+-------+-------------+--------------------------+
|T001          |101        |Bangkok   |2026-05-01      |1200.5|success|pos          |2026-06-01 15:34:00.965035|
|T002          |102        |Bangkok   |2026-05-01      |250.0 |success|mobile_app   |2026-06-01 15:34:00.965035|
|T003          |103        |Chiang Mai|2026-05-02      |3400.0|success|pos          |2026-06-01 15:34:00.965035|
|T004          |104        |Rayong    |2026-05-02      |0.0   |failed |web          |2026-06-01 15:34:00.965035|
|T005          |105        |Bangkok   |2026-05-03      |780.0 |pending|mobile_app   |2026-06-01 15:34:00.965035|
|T006          |106        |Phuket    |2026-05-03      |5600.0|success|partner_api  |2026-06-01 

## PySpark code examples

ใน section นี้เราจะเขียน dataset เดียวกันออกเป็น 2 แบบ:

1. Parquet files ที่อ่านกลับด้วย file path
2. Delta Lakehouse table ที่อ่านกลับด้วย table name

ให้สังเกตว่า code สำหรับอ่านข้อมูลต่างกัน และ mindset ที่ตามมาก็ต่างกันด้วย


### Example 1: Define paths and table names

ใน Fabric Lakehouse path ฝั่ง file มักอยู่ใต้ `Files/...` ส่วน table จะถูก register เป็น table name ใน Lakehouse


In [3]:
base_file_path = "Files/day13_lakehouse_table_delta_parquet"
parquet_path = f"{base_file_path}/transactions_parquet"

delta_table_name = "day13_transactions_delta"
converted_table_name = "day13_transactions_from_parquet"

print("Parquet path:", parquet_path)
print("Delta table name:", delta_table_name)
print("Converted table name:", converted_table_name)

StatementMeta(, c54a50f4-369a-4062-9e9f-45ea83d8c97b, 5, Finished, Available, Finished, False)

Parquet path: Files/day13_lakehouse_table_delta_parquet/transactions_parquet
Delta table name: day13_transactions_delta
Converted table name: day13_transactions_from_parquet


### Example 2: Write and read Parquet files by path

ตัวอย่างนี้เขียนข้อมูลไปที่ Parquet path แล้วอ่านกลับด้วย `spark.read.parquet()`

สังเกตว่าเราไม่ได้สร้าง table name อะไรขึ้นมา เราแค่สร้าง files ที่ path หนึ่งใน Lakehouse Files area


In [4]:
(
    df_transactions
    .write
    .mode("overwrite")
    .parquet(parquet_path)
)

df_parquet = spark.read.parquet(parquet_path)

df_parquet.show(truncate=False)
print("Parquet row count:", df_parquet.count())

StatementMeta(, c54a50f4-369a-4062-9e9f-45ea83d8c97b, 6, Finished, Available, Finished, False)

+--------------+-----------+----------+----------------+------+-------+-------------+--------------------------+
|transaction_id|customer_id|region    |transaction_date|amount|status |source_system|ingestion_timestamp       |
+--------------+-----------+----------+----------------+------+-------+-------------+--------------------------+
|T002          |102        |Bangkok   |2026-05-01      |250.0 |success|mobile_app   |2026-06-01 15:37:02.416139|
|T005          |105        |Bangkok   |2026-05-03      |780.0 |pending|mobile_app   |2026-06-01 15:37:02.416139|
|T006          |106        |Phuket    |2026-05-03      |5600.0|success|partner_api  |2026-06-01 15:37:02.416139|
|T003          |103        |Chiang Mai|2026-05-02      |3400.0|success|pos          |2026-06-01 15:37:02.416139|
|T008          |108        |Chiang Mai|2026-05-04      |980.0 |success|pos          |2026-06-01 15:37:02.416139|
|T001          |101        |Bangkok   |2026-05-01      |1200.5|success|pos          |2026-06-01 

### Example 3: Query data from Parquet path

Parquet path ใช้กับ DataFrame API ได้ดี แต่ถ้าจะใช้ SQL ตรง ๆ เราต้อง register เป็น temp view ก่อน หรือสร้าง table เพิ่มเอง


In [5]:
df_parquet_summary = (
    df_parquet
    .groupBy("status")
    .agg(
        F.count("*").alias("transaction_count"),
        F.round(F.sum("amount"), 2).alias("total_amount")
    )
    .orderBy("status")
)

df_parquet_summary.show(truncate=False)

StatementMeta(, c54a50f4-369a-4062-9e9f-45ea83d8c97b, 7, Finished, Available, Finished, False)

+-------+-----------------+------------+
|status |transaction_count|total_amount|
+-------+-----------------+------------+
|failed |2                |150.0       |
|pending|1                |780.0       |
|success|5                |11430.5     |
+-------+-----------------+------------+



### Example 4: Write a Delta Lakehouse table with saveAsTable

ตัวอย่างนี้เขียนข้อมูลเป็น Delta table และ register table name ใน Lakehouse

ใน Fabric learning lab ให้ใช้ `format("delta")` ให้ชัดเจน เพื่อสื่อว่า table นี้ตั้งใจให้เป็น Delta table ไม่ใช่แค่ Parquet files ธรรมดา


In [6]:
(
    df_transactions
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(delta_table_name)
)

print(f"Created Delta table: {delta_table_name}")

StatementMeta(, c54a50f4-369a-4062-9e9f-45ea83d8c97b, 8, Finished, Available, Finished, False)

Created Delta table: day13_transactions_delta


### Example 5: Read Delta Lakehouse table by table name

เมื่อ table ถูกสร้างแล้ว เราสามารถอ่านด้วย `spark.read.table()` ได้ ไม่ต้องจำ physical path เอง


In [7]:
df_delta_table = spark.read.table(delta_table_name)

df_delta_table.show(truncate=False)
df_delta_table.printSchema()
print("Delta table row count:", df_delta_table.count())

StatementMeta(, c54a50f4-369a-4062-9e9f-45ea83d8c97b, 9, Finished, Available, Finished, False)

+--------------+-----------+----------+----------------+------+-------+-------------+--------------------------+
|transaction_id|customer_id|region    |transaction_date|amount|status |source_system|ingestion_timestamp       |
+--------------+-----------+----------+----------------+------+-------+-------------+--------------------------+
|T006          |106        |Phuket    |2026-05-03      |5600.0|success|partner_api  |2026-06-01 15:39:09.517359|
|T002          |102        |Bangkok   |2026-05-01      |250.0 |success|mobile_app   |2026-06-01 15:39:09.517359|
|T005          |105        |Bangkok   |2026-05-03      |780.0 |pending|mobile_app   |2026-06-01 15:39:09.517359|
|T003          |103        |Chiang Mai|2026-05-02      |3400.0|success|pos          |2026-06-01 15:39:09.517359|
|T008          |108        |Chiang Mai|2026-05-04      |980.0 |success|pos          |2026-06-01 15:39:09.517359|
|T001          |101        |Bangkok   |2026-05-01      |1200.5|success|pos          |2026-06-01 

### Example 6: Query Delta table with Spark SQL

ข้อดีของ table คือ query ด้วย SQL ได้ตรงจาก table name ซึ่งช่วยให้ logic อ่านง่ายขึ้นสำหรับงาน validation หรือ ad-hoc analysis


In [8]:
spark.sql(f"""
    SELECT
        status,
        COUNT(*) AS transaction_count,
        ROUND(SUM(amount), 2) AS total_amount
    FROM {delta_table_name}
    GROUP BY status
    ORDER BY status
""").show(truncate=False)

StatementMeta(, c54a50f4-369a-4062-9e9f-45ea83d8c97b, 10, Finished, Available, Finished, False)

+-------+-----------------+------------+
|status |transaction_count|total_amount|
+-------+-----------------+------------+
|failed |2                |150.0       |
|pending|1                |780.0       |
|success|5                |11430.5     |
+-------+-----------------+------------+



### Example 7: Inspect Delta table detail

`DESCRIBE DETAIL` ช่วยให้เห็น metadata ระดับ table เช่น format, location, จำนวน files และขนาดข้อมูลของ table

In [10]:
spark.sql(f"DESCRIBE DETAIL {delta_table_name}").show(truncate=False)

StatementMeta(, c54a50f4-369a-4062-9e9f-45ea83d8c97b, 12, Finished, Available, Finished, False)

+------+------------------------------------+-----------------------------------------------------------+-----------+--------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------+-----------------------+----------------+-----------------+--------+-----------+---------------------------------------------------------------------------+----------------+----------------+------------------------+
|format|id                                  |name                                                       |description|location                                                                                                                                          |createdAt              |lastModified           |partitionColumns|clusteringColumns|numFiles|sizeInBytes|properties                                                                 |minReaderVersion|minWriterVersion|tableFeatures     

### Example 8: Optional inspect _delta_log folder

Delta table จะมี `_delta_log` อยู่ใต้ table location เพื่อเก็บ transaction log


In [26]:
table_detail = spark.sql(f"DESCRIBE DETAIL {delta_table_name}").select("location").first()  # first() returns the first metadata row as a Python Row object.

print("Delta table detail:", table_detail)

# Extract the location value from the Row object.
delta_location = table_detail["location"]
delta_log_path = delta_location.rstrip("/") + "/_delta_log"

print("\nDelta table location:", delta_location)
print("Delta log path:", delta_log_path)

StatementMeta(, c54a50f4-369a-4062-9e9f-45ea83d8c97b, 28, Finished, Available, Finished, False)

Delta table detail: Row(location='abfss://a5de4c5d-a2ed-4a31-923d-ff4fcd64006e@onelake.dfs.fabric.microsoft.com/8f868c6a-7fc0-4ade-a075-39cee1694ad9/Tables/day13_transactions_delta')

Delta table location: abfss://a5de4c5d-a2ed-4a31-923d-ff4fcd64006e@onelake.dfs.fabric.microsoft.com/8f868c6a-7fc0-4ade-a075-39cee1694ad9/Tables/day13_transactions_delta
Delta log path: abfss://a5de4c5d-a2ed-4a31-923d-ff4fcd64006e@onelake.dfs.fabric.microsoft.com/8f868c6a-7fc0-4ade-a075-39cee1694ad9/Tables/day13_transactions_delta/_delta_log


### Example 9: Convert Parquet path output into a Lakehouse table

บางครั้งเราอาจมี Parquet files อยู่ก่อน แล้วค่อยอ่านกลับมาเขียนเป็น Delta table เพื่อให้ downstream ใช้ table name ได้

ตัวอย่างนี้ไม่ได้แปลว่า Parquet path แย่เสมอไป แต่แสดงให้เห็นว่า curated output ควรถูก promote เป็น table เมื่อเริ่มถูกใช้ต่อจริงจัง


In [13]:
df_from_parquet = (
    spark.read.parquet(parquet_path)
    .withColumn("curation_note", F.lit("converted_from_parquet_path"))
)

(
    df_from_parquet
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(converted_table_name)
)

spark.read.table(converted_table_name).select(
    "transaction_id",
    "status",
    "amount",
    "curation_note"
).show(truncate=False)

StatementMeta(, c54a50f4-369a-4062-9e9f-45ea83d8c97b, 15, Finished, Available, Finished, False)

+--------------+-------+------+---------------------------+
|transaction_id|status |amount|curation_note              |
+--------------+-------+------+---------------------------+
|T006          |success|5600.0|converted_from_parquet_path|
|T002          |success|250.0 |converted_from_parquet_path|
|T005          |pending|780.0 |converted_from_parquet_path|
|T003          |success|3400.0|converted_from_parquet_path|
|T008          |success|980.0 |converted_from_parquet_path|
|T001          |success|1200.5|converted_from_parquet_path|
|T007          |failed |150.0 |converted_from_parquet_path|
|T004          |failed |0.0   |converted_from_parquet_path|
+--------------+-------+------+---------------------------+



### Example 10: Compare file path access vs table name access

ตัวอย่างนี้สรุป mindset ให้เห็นชัด ๆ ว่า path-based read กับ table-based read ต่างกันตรง interface ที่เราใช้เข้าถึงข้อมูล


In [27]:
# File-based access: you must know the storage path
path_based_count = spark.read.parquet(parquet_path).count()

# Table-based access: you use the table name registered in Lakehouse
name_based_count = spark.read.table(delta_table_name).count()

print("Path-based Parquet count:", path_based_count)
print("Table-based Delta count:", name_based_count)

StatementMeta(, c54a50f4-369a-4062-9e9f-45ea83d8c97b, 29, Finished, Available, Finished, False)

Path-based Parquet count: 8
Table-based Delta count: 8


## Quick recap

| Question | Answer |
|---|---|
| Parquet คืออะไร? | Columnar file format สำหรับ analytics data |
| Delta table คืออะไร? | Table format ที่ใช้ Parquet data files และมี `_delta_log` เพิ่มเข้ามา |
| อ่าน Parquet path ยังไง? | ใช้ `spark.read.parquet("Files/...")` |
| อ่าน Lakehouse table ยังไง? | ใช้ `spark.read.table("table_name")` หรือ `spark.sql()` |
| ทำไม table ดีกว่า file สำหรับ curated data? | เพราะมี table name, metadata และ transaction log concept ช่วยให้ใช้งานต่อได้ reliable กว่า |
| Day 13 ควรจำอะไรที่สุด? | File path คือ storage access ส่วน table คือ data product/interface ที่ consumer ใช้ต่อได้ง่ายกว่า |


## Exercises


### Exercise 1: Write a small product dataset as Parquet files

สร้าง DataFrame ชื่อ `df_products` แล้วเขียนเป็น Parquet path ใต้ `Files/day13_lakehouse_table_delta_parquet/products_parquet`

Requirements:

- มี columns: `product_id`, `product_name`, `category`, `price`
- เขียนเป็น Parquet ด้วย `mode("overwrite")`
- อ่านกลับด้วย `spark.read.parquet()`
- แสดงผลด้วย `.show()` และ `.printSchema()`

Expected result:

- อ่านข้อมูลกลับจาก path ได้
- row count เท่ากับจำนวน mock products ที่สร้างไว้


In [28]:
products_data = [
    ("P001", "Basic Plan", "subscription", 199.00),
    ("P002", "Premium Plan", "subscription", 499.00),
    ("P003", "Installation Service", "service", 1200.00),
    ("P004", "Support Package", "service", 800.00),
]

product_schema = T.StructType([
    T.StructField("product_id", T.StringType(), False),
    T.StructField("product_name", T.StringType(), True),
    T.StructField("category", T.StringType(), True),
    T.StructField("price", T.DoubleType(), True),
])

df_products = spark.createDataFrame(products_data, product_schema)

products_parquet_path = f"{base_file_path}/products_parquet"

(
    df_products
    .write
    .mode("overwrite")
    .parquet(products_parquet_path)
)

df_products_from_path = spark.read.parquet(products_parquet_path)

df_products_from_path.show(truncate=False)
df_products_from_path.printSchema()
print("Product count:", df_products_from_path.count())

StatementMeta(, c54a50f4-369a-4062-9e9f-45ea83d8c97b, 30, Finished, Available, Finished, False)

+----------+--------------------+------------+------+
|product_id|product_name        |category    |price |
+----------+--------------------+------------+------+
|P003      |Installation Service|service     |1200.0|
|P002      |Premium Plan        |subscription|499.0 |
|P001      |Basic Plan          |subscription|199.0 |
|P004      |Support Package     |service     |800.0 |
+----------+--------------------+------------+------+

root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)

Product count: 4


### Exercise 2: Promote product Parquet files into a Delta table

อ่าน product Parquet files จาก Exercise 1 แล้วเขียนเป็น Delta Lakehouse table ชื่อ `day13_products_delta`

Requirements:

- อ่านจาก `products_parquet_path`
- เพิ่ม column `is_active = true`
- เขียนเป็น Delta table ด้วย `format("delta")` และ `saveAsTable()`
- อ่านกลับด้วย `spark.read.table()`

Expected result:

- มี table `day13_products_delta`
- อ่าน table ด้วย table name ได้
- เห็น column `is_active`


In [29]:
products_delta_table_name = "day13_products_delta"

products_for_table = (
    spark.read.parquet(products_parquet_path)
    .withColumn("is_active", F.lit(True))
)

(
    products_for_table
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(products_delta_table_name)
)

df_products_delta = spark.read.table(products_delta_table_name)

df_products_delta.show(truncate=False)
print("Products Delta table count:", df_products_delta.count())

StatementMeta(, c54a50f4-369a-4062-9e9f-45ea83d8c97b, 31, Finished, Available, Finished, False)

+----------+--------------------+------------+------+---------+
|product_id|product_name        |category    |price |is_active|
+----------+--------------------+------------+------+---------+
|P003      |Installation Service|service     |1200.0|true     |
|P002      |Premium Plan        |subscription|499.0 |true     |
|P004      |Support Package     |service     |800.0 |true     |
|P001      |Basic Plan          |subscription|199.0 |true     |
+----------+--------------------+------------+------+---------+

Products Delta table count: 4


### Exercise 3: Query the Delta table with Spark SQL

ใช้ Spark SQL query table `day13_products_delta` เพื่อสรุปจำนวน product และราคาเฉลี่ยตาม `category`

Requirements:

- ใช้ `spark.sql()`
- group by `category`
- output columns: `category`, `product_count`, `avg_price`
- ใช้ `ROUND` หรือ `F.round` ให้ราคาเฉลี่ยอ่านง่าย

Expected result:

- ได้ summary ระดับ category
- Query อ้างอิง table name ไม่ใช่ file path


In [30]:
spark.sql(f"""
    SELECT
        category,
        COUNT(*) AS product_count,
        ROUND(AVG(price), 2) AS avg_price
    FROM {products_delta_table_name}
    GROUP BY category
    ORDER BY category
""").show(truncate=False)

StatementMeta(, c54a50f4-369a-4062-9e9f-45ea83d8c97b, 32, Finished, Available, Finished, False)

+------------+-------------+---------+
|category    |product_count|avg_price|
+------------+-------------+---------+
|service     |2            |1000.0   |
|subscription|2            |349.0    |
+------------+-------------+---------+



## Common mistakes

1. **คิดว่า Parquet path คือ Lakehouse table เสมอ**

   Parquet files เป็น files ที่ path หนึ่ง แต่ table ต้องมี metadata / table name ให้ query ผ่าน catalog หรือ Lakehouse table list ได้

2. **ใช้ path กระจายอยู่หลาย notebook โดยไม่จำเป็น**

   ถ้า dataset ถูกใช้ต่อหลายจุด การใช้ table name จะ maintain ง่ายกว่า hardcode path ซ้ำ ๆ

3. **คิดว่า Delta คือ Parquet อีกชื่อหนึ่งเท่านั้น**

   Delta table ใช้ Parquet data files จริง แต่มี `_delta_log` และ table behavior เพิ่มขึ้นมา จึงไม่ควรมองว่าเหมือน Parquet path ธรรมดา 100%

4. **คาดหวังว่า Parquet files จะถูกเห็นเป็น SQL table อัตโนมัติ**

   ถ้าต้องการให้ consumer query เป็น table ควรสร้าง/register เป็น Lakehouse table ให้ชัดเจน ไม่ใช่แค่โยน Parquet files ไว้ใน folder

5. **ใช้ `overwrite` กับ table/path ที่มีข้อมูลสำคัญโดยไม่ตั้งใจ**

   ใน lab ใช้ `overwrite` เพื่อให้ rerun ง่าย แต่ production ต้องระวังมาก เพราะอาจลบหรือแทนที่ข้อมูลเดิมผิดชุดได้


## Expected Output / Success Criteria

เมื่อจบ Day 13 ควรทำได้:

- อธิบายความต่างระหว่าง Parquet files, file path access และ Lakehouse table ได้
- อธิบายได้ว่า Delta table ใช้ Parquet data files ร่วมกับ `_delta_log` และ table metadata
- เขียนข้อมูลเป็น Parquet path ด้วย `.write.parquet()` ได้
- อ่านข้อมูลกลับจาก Parquet path ด้วย `spark.read.parquet()` ได้
- เขียนข้อมูลเป็น Delta Lakehouse table ด้วย `.format("delta").saveAsTable()` ได้
- อ่าน Delta Lakehouse table ด้วย `spark.read.table()` และ query ด้วย `spark.sql()` ได้
- ใช้ `DESCRIBE DETAIL` เพื่อตรวจ table metadata ได้
- เข้าใจว่า curated output ที่ถูกใช้ต่อควรเริ่มคิดเป็น table ไม่ใช่ file อย่างเดียว


## Final takeaway

Day 13 ไม่ได้ต้องการให้จำ syntax เยอะ แต่ต้องการเปลี่ยน mindset จาก “เขียนไฟล์ออกไปที่ path” เป็น “สร้าง table ที่ downstream ใช้ต่อได้อย่างน่าเชื่อถือกว่า”

> Parquet เหมาะกับการเก็บข้อมูลเป็น files แต่ Delta Lakehouse table ทำให้ข้อมูลกลายเป็น table ที่มี metadata, transaction log และ interface ที่อ่านด้วย table name ได้

Mindset ที่ควรจำ:

- Raw/temporary output อาจเริ่มจาก file path ได้
- Curated/reusable output ควรถูก promote เป็น Lakehouse table
- Delta table คือ Parquet data files + `_delta_log` + table metadata
- Table name ช่วยลดการ hardcode path และทำให้ notebook/pipeline อ่านง่ายขึ้น
- ก่อน overwrite table หรือ path ต้องแน่ใจว่าเป็น lab/test data ไม่ใช่ข้อมูลสำคัญ


## Optional cleanup

In [ ]:
# spark.sql("DROP TABLE IF EXISTS day13_transactions_delta")
# spark.sql("DROP TABLE IF EXISTS day13_transactions_from_parquet")
# spark.sql("DROP TABLE IF EXISTS day13_products_delta")

# Optional: remove files from Lakehouse Files if needed
# notebookutils.fs.rm("Files/day13_lakehouse_table_delta_parquet", True)